# Space Invaders from Scratch × Rainbow DQN

**Team:** Sergey Peshkov · Anton Kropotov · Arina Nikolaeva — Reinforcement Learning course, Innopolis University

| Part | Owner |
|---|---|
| 1. Environment: states, actions, rewards, goal | Sergey |
| 2. Rainbow DQN and its six components | Arina |
| 3. Worked example: one Rainbow update by hand | Arina |
| 4. Implementation, training, loading the model | Anton |
| 5. Results: trained agent vs random, demo | Sergey |

Run this notebook from the project root (or from `notebooks/`). Training itself is done with `train.py`
(locally or with `notebooks/kaggle_train.ipynb`); this notebook loads the results.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage
from space_invaders import SpaceInvadersEnv, GameConfig, DIFFICULTY, EVENT_NAMES, ACTION_NAMES, REWARD_PRESETS
RUN = "runs/rainbow_pixels"   # <-- training run to analyse in part 5

## 1. Environment (Sergey)

We re-implemented Space Invaders in pure Python/NumPy (`space_invaders/game.py`) and wrapped it in a
Gymnasium-compatible environment (`space_invaders/env.py`). Writing the game ourselves lets us
(1) read every game event and turn it into a reward **trigger**, (2) change the difficulty, and
(3) give the agent either pixels or the exact game state.

**Goal of the agent:** maximise the game score — destroy invaders (10–30 points) and the UFO (100),
without losing the 3 lives and before the invaders reach the ground.

### MDP

| | Definition |
|---|---|
| **State** $s_t$ | *pixels*: last 4 grayscale frames, 4×84×84 uint8 (frame stack makes the state Markov: speed and direction are visible) · *vector*: 75 numbers — player x, own bullet, formation position/direction/timer, 4 nearest enemy bullets, UFO, lives, shield health, alive-mask of invaders (6×8, zero-padded) |
| **Actions** $a_t$ | 6 discrete actions, same as Atari: NOOP, FIRE, RIGHT, LEFT, RIGHT+FIRE, LEFT+FIRE |
| **Transition** | one agent step = 4 game frames (frame skip). Random invader fire and UFO ⇒ stochastic. Episodes start with 0–30 random no-op frames |
| **Reward** $r_t$ | weighted sum of the events (triggers) of these 4 frames, see below; clipped to [−1, 1] during training |
| **Termination** | all lives lost or invaders reach the player row; truncation after 10 000 steps |
| **Discount** | γ = 0.99 |

In [ ]:
env = SpaceInvadersEnv(obs_mode="pixels", seed=0)
obs, info = env.reset(seed=0)
for _ in range(40):
    obs, r, term, trunc, info = env.step(4)   # RIGHT+FIRE
print("observation:", obs.shape, obs.dtype, "| actions:", ACTION_NAMES)
fig, ax = plt.subplots(1, 5, figsize=(16, 3.4))
ax[0].imshow(env.render(scale=2)); ax[0].set_title("game (render)")
for i in range(4):
    ax[i + 1].imshow(obs[i], cmap="gray"); ax[i + 1].set_title(f"agent input, frame t-{3 - i}")
for a in ax: a.axis("off")
plt.show()
vec = SpaceInvadersEnv(obs_mode="vector").reset(seed=0)[0]
print("vector observation size:", vec.shape)

### Reward triggers

The game reports these events every frame (`info["events"]`). A reward preset is a set of weights:
$r_t = \sum_k w_k \cdot \text{event}_k$.

* `score` — the classic reward: game score (after clipping: +1 for any kill).
* `shaped` — denser feedback: also penalises being hit / missing and rewards dodging and clearing a wave.

In [ ]:
import pandas as pd
rows = [{"event": e, **{p: REWARD_PRESETS[p].get(e, 0) for p in REWARD_PRESETS}} for e in EVENT_NAMES]
display(pd.DataFrame(rows))
lv = pd.DataFrame({lvl: {**GameConfig().to_dict(), **DIFFICULTY[lvl]} for lvl in DIFFICULTY}).loc[
    ["inv_rows", "inv_cols", "inv_move_interval", "inv_fire_prob", "max_enemy_bullets", "enemy_bullet_speed"]]
lv.columns = ["level 0 (easy)", "level 1 (normal)", "level 2 (hard)"]
display(lv)

### Random-agent baseline (reference for the results)

In [ ]:
from rainbow.evaluation import run_episodes, random_policy, summarize
for lvl in (0, 1, 2):
    rows = run_episodes(random_policy(6, 0), SpaceInvadersEnv(obs_mode="vector", level=lvl), 20, seed=12345)
    s = summarize(rows)
    print(f"level {lvl}: random score {s['mean']:.0f} ± {s['std']:.0f}, mean length {np.mean([r['length'] for r in rows]):.0f} steps")

## 2. Rainbow DQN (Arina)

**DQN** (Mnih et al., 2015) learns $Q(s,a)$ with a neural network by minimising
$\big(r + \gamma \max_{a'} Q_{\bar\theta}(s',a') - Q_\theta(s,a)\big)^2$
on minibatches from an **experience replay** buffer, using a periodically copied **target network** $\bar\theta$.

**Rainbow** (Hessel et al., 2018) combines six independent improvements of DQN:

| # | Component | Problem it solves | What changes |
|---|---|---|---|
| 1 | **Double Q-learning** | $\max$ over noisy estimates overestimates Q | online net *chooses* $a^*=\arg\max_{a'}Q_\theta(s',a')$, target net *evaluates* it |
| 2 | **Prioritized replay** | uniform sampling wastes updates on already-learned transitions | sample transition $i$ with $P(i)\propto p_i^{\alpha}$, $p_i$ = last loss; correct the bias with weights $w_i=(N P(i))^{-\beta}$, β annealed 0.4 → 1 |
| 3 | **Dueling network** | in many states the action does not matter | two streams: $Q(s,a)=V(s)+A(s,a)-\frac{1}{|\mathcal A|}\sum_{a'}A(s,a')$ |
| 4 | **Multi-step returns** | 1-step targets propagate reward slowly | $R_t^{(n)}=\sum_{k=0}^{n-1}\gamma^k r_{t+k}$, target uses $\gamma^n$ and $s_{t+n}$ (n = 3) |
| 5 | **Distributional RL (C51)** | the mean hides the uncertainty of the return | predict a distribution over 51 fixed atoms $z_i\in[-10,10]$; loss = cross-entropy between the projected target distribution and the prediction |
| 6 | **Noisy Nets** | ε-greedy explores the same way in every state | linear layers $y=(\mu_w+\sigma_w\odot\varepsilon_w)x+(\mu_b+\sigma_b\odot\varepsilon_b)$, σ is learned, so the agent learns *how much* to explore |

### The Rainbow loss
For a sampled transition $(s_t, a_t, R^{(n)}_t, s_{t+n})$:

1. $a^* = \arg\max_{a} \sum_i z_i\, p_\theta(s_{t+n}, a)_i$ (online net — Double Q)
2. target atoms $\hat T z_i = R^{(n)}_t + \gamma^n z_i$ (0 if terminal), clipped to $[V_{min}, V_{max}]$, with probabilities $p_{\bar\theta}(s_{t+n}, a^*)_i$
3. project them back onto the fixed atoms → $m$ (each atom's probability is split between its two neighbours)
4. $\mathcal L = -\sum_i m_i \log p_\theta(s_t, a_t)_i$, weighted by $w_i$ (PER); the new priority is $\mathcal L$.

What the original paper found (Fig. 3, ablations over 57 Atari games): removing **prioritized replay** or
**multi-step** hurts most, then **distributional**; **noisy nets** help on some games; **dueling** and
**double** matter least once the other parts are in. *Revisiting Rainbow* (Obando-Ceron & Castro, 2021)
shows that the importance of each component depends on the environment — so an ablation on our game
is a meaningful experiment.

## 3. Worked example: one Rainbow update by hand (Arina)

Tiny numbers so every step can be checked with a calculator: **5 atoms** $z = [-2,-1,0,1,2]$ ($\Delta z = 1$),
**2 actions**, $n=3$, $\gamma = 0.9$.

The sampled transition: rewards $r_t, r_{t+1}, r_{t+2} = 1, 0, 1$, the episode did not end.

In [ ]:
z = np.array([-2., -1., 0., 1., 2.]); v_min, v_max, dz = -2., 2., 1.
gamma, n = 0.9, 3
rewards = [1., 0., 1.]

# Step 1. n-step return
R = sum(gamma**k * r for k, r in enumerate(rewards))
print(f"R = 1 + 0.9*0 + 0.81*1 = {R:.3f},   gamma^n = {gamma**n:.3f}")

In [ ]:
# Step 2. Double Q: the ONLINE net chooses the next action at s_{t+3}
p_online_next = np.array([[0.10, 0.20, 0.40, 0.20, 0.10],    # action 0
                          [0.00, 0.10, 0.20, 0.40, 0.30]])   # action 1
q_online_next = p_online_next @ z
a_star = int(q_online_next.argmax())
print("Q_online(s', a) =", q_online_next, "-> a* =", a_star)

# ... and the TARGET net gives the distribution for that action
p_target_next = np.array([[0.20, 0.30, 0.30, 0.10, 0.10],
                          [0.05, 0.15, 0.30, 0.30, 0.20]])
p_next = p_target_next[a_star]
print("target distribution p(s', a*) =", p_next)

In [ ]:
# Step 3. Shift and shrink the atoms: Tz = R + gamma^n * z, clipped to [v_min, v_max]
Tz = np.clip(R + gamma**n * z, v_min, v_max)
b = (Tz - v_min) / dz                     # position between atoms
l, u = np.floor(b).astype(int), np.ceil(b).astype(int)
# exact hit on an atom (l == u): move one index so the mass is not lost
l[(u > 0) & (l == u)] -= 1
u[(l < len(z) - 1) & (l == u)] += 1
m = np.zeros(len(z))
for j in range(len(z)):
    m[l[j]] += p_next[j] * (u[j] - b[j])
    m[u[j]] += p_next[j] * (b[j] - l[j])
display(pd.DataFrame({"z_j": z, "p(s',a*)_j": p_next, "Tz_j": Tz.round(3), "b_j": b.round(3),
                      "l": l, "u": u, "to l": (p_next * (u - b)).round(4), "to u": (p_next * (b - l)).round(4)}))
print("projected target m =", m.round(4), " sum =", m.sum().round(6))
print(f"mean of m = {m @ z:.3f}  (unclipped target mean would be R + g^n E[z] = {R + gamma**n * (p_next @ z):.3f}; atoms above 2 were clipped)")

In [ ]:
# Step 4. Cross-entropy loss against the ONLINE prediction for (s_t, a_t)
logits = np.log(np.array([0.10, 0.20, 0.30, 0.25, 0.15]))   # pretend these are the network outputs
p_pred = np.exp(logits) / np.exp(logits).sum()
loss = -(m * np.log(p_pred)).sum()
print("prediction p(s,a) =", p_pred.round(3), " Q =", (p_pred @ z).round(3))
print(f"loss = -sum m_i log p_i = {loss:.4f}")
print("gradient w.r.t. logits = p - m =", (p_pred - m).round(4), "-> gradient descent moves probability mass to z = 2, where the target m is concentrated")

# Step 5. Prioritized replay: new priority and importance weight
alpha, beta, N, P_i = 0.5, 0.4, 100_000, 2e-5
print(f"new priority p_i = loss^alpha = {loss**alpha:.4f}")
print(f"IS weight (before normalising by max) = (N*P(i))^-beta = {(N * P_i) ** -beta:.4f}")

# Step 6. Same step taken by the real implementation (needs torch)
try:
    import torch
    from rainbow.agent import project_distribution
    m_t = project_distribution(torch.tensor(p_next[None], dtype=torch.float32), torch.tensor([R]), torch.tensor([1.0]),
                               gamma**n, torch.tensor(z, dtype=torch.float32), v_min, v_max)
    print("rainbow/agent.py gives m =", m_t.numpy().round(4), "-> matches:", np.allclose(m_t.numpy(), m, atol=1e-5))
except ImportError:
    print("(torch not installed - skipping the comparison with rainbow/agent.py)")

**Without the distributional part** the same step would be a scalar target:
$y = R + \gamma^n\,Q_{\bar\theta}(s', a^*) = 1.81 + 0.729 \cdot Q_{\bar\theta}(s',1)$ and a Huber loss on $y - Q_\theta(s,a)$.

In [ ]:
q_target = p_target_next[a_star] @ z
print(f"scalar target y = {R:.3f} + {gamma**n:.3f} * {q_target:.2f} = {R + gamma**n * q_target:.3f}")

## 4. Implementation and training (Anton)

```
space_invaders/   game.py (rules, events, drawing)  env.py (Gymnasium env, triggers, difficulty)
rainbow/          model.py (CNN/MLP torso, NoisyLinear, dueling, C51 head)
                  memory.py (prioritized n-step replay, sum tree, frame-stack reconstruction)
                  agent.py (Rainbow update, ablation switches)   envs.py (our game or Atari ALE)
train.py  evaluate.py  demo.py  play.py
```

| Hyper-parameter | Value | | Hyper-parameter | Value |
|---|---|---|---|---|
| network | Nature CNN, 512 hidden (MLP 256 for vectors) | | n-step | 3 |
| atoms / support | 51 / [−10, 10] | | γ | 0.99 |
| optimizer | Adam, lr 1e-4 (paper 6.25e-5), ε 1.5e-4 | | batch | 32, update every 4 steps |
| replay | 200k (paper 1M), PER α = 0.5, β 0.4 → 1 | | target update | every 2 000 steps (paper 8 000) |
| learning starts | 10 000 steps | | noisy σ₀ | 0.5 |

```bash
python train.py --id rainbow_pixels --steps 1000000          # main run
python train.py --id dqn --dqn --steps 1000000               # vanilla DQN baseline
python train.py --id shaped --reward shaped --steps 1000000  # reward-trigger study
python train.py --id rainbow_pixels --steps 1000000 --resume # continue after interruption
```
Each run writes `runs/<id>/`: `config.json`, `episodes.csv`, `eval.csv`, `learning_curve.png`,
`model.pth` (best evaluation), `final.pth`, `checkpoint.pt`.

In [ ]:
# Load the trained model and let it play one episode
try:
    from rainbow import RainbowAgent
    agent, meta = RainbowAgent.from_file(f"{RUN}/model.pth")
    print(f"loaded model from step {meta.get('step')}, evaluation score {meta.get('eval_score')}")
    env = SpaceInvadersEnv(obs_mode="pixels" if agent.pixel_input else "vector")
    rows = run_episodes(lambda o: agent.act(o, greedy=True), env, 1, seed=7)
    print("one episode:", rows[0])
except (ImportError, FileNotFoundError) as e:
    print("No trained model yet (or torch missing):", e)

## 5. Results (Sergey)

### Learning curve

In [ ]:
if os.path.exists(f"{RUN}/eval.csv"):
    ev = pd.read_csv(f"{RUN}/eval.csv"); ep = pd.read_csv(f"{RUN}/episodes.csv")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(ep["step"], ep["score"].rolling(20, min_periods=1).mean(), color="#9AA3C0", label="training episodes (avg 20)")
    ax.plot(ev["step"], ev["mean_score"], "o-", color="#F72585", label="evaluation, 10 episodes")
    ax.fill_between(ev["step"], ev["mean_score"] - ev["std_score"], ev["mean_score"] + ev["std_score"], color="#F72585", alpha=.15)
    ax.axhline(ev["random_score"].iloc[0], ls="--", color="#4CC9F0", label="random agent")
    ax.set_xlabel("agent steps (×4 frames)"); ax.set_ylabel("game score"); ax.grid(alpha=.3); ax.legend(); plt.show()
    display(ev.tail())
else:
    print(f"{RUN}/eval.csv not found - train first")

### Trained agent vs random agent
Same 30 seeds for both agents (`evaluate.py` does the same from the command line and saves `results/`).

In [ ]:
N_EPISODES = 30
try:
    from rainbow import RainbowAgent
    agent, meta = RainbowAgent.from_file(f"{RUN}/model.pth")
    obs_mode = "pixels" if agent.pixel_input else "vector"
    res = {"Random": run_episodes(random_policy(6, 0), SpaceInvadersEnv(obs_mode=obs_mode), N_EPISODES, seed=12345),
           "Rainbow": run_episodes(lambda o: agent.act(o, greedy=True), SpaceInvadersEnv(obs_mode=obs_mode), N_EPISODES, seed=12345)}
    table = []
    for name, rows in res.items():
        s = summarize(rows)
        kills = sum(r.get("ev_enemy_killed", 0) + r.get("ev_ufo_killed", 0) for r in rows)
        shots = sum(r.get("ev_shot_fired", 0) for r in rows)
        table.append({"agent": name, "mean score": s["mean"], "std": s["std"], "median": s["median"], "max": s["max"],
                      "mean length": np.mean([r["length"] for r in rows]), "accuracy": kills / max(1, shots),
                      "waves cleared": np.mean([r.get("waves_cleared", 0) for r in rows])})
    table = pd.DataFrame(table).round(2); display(table)
    plt.figure(figsize=(5, 4)); plt.boxplot([[r["score"] for r in rows] for rows in res.values()], showmeans=True)
    plt.xticks([1, 2], list(res)); plt.ylabel("game score"); plt.title(f"{N_EPISODES} episodes, same seeds"); plt.grid(alpha=.3); plt.show()
    from scipy.stats import mannwhitneyu
    print("Mann-Whitney U test, Rainbow > Random: p =", mannwhitneyu([r["score"] for r in res["Rainbow"]], [r["score"] for r in res["Random"]], alternative="greater").pvalue)
except (ImportError, FileNotFoundError) as e:
    print("Need a trained model and torch:", e)

### Demo video
`python demo.py --model runs/rainbow_pixels/model.pth --compare-random --out demo_vs_random.mp4`
records the agent (left) next to a random agent (right) with the reward triggers shown on screen.

In [ ]:
from IPython.display import Video
if os.path.exists("demo_vs_random.mp4"):
    display(Video("demo_vs_random.mp4", embed=True, width=720))

### Reference numbers and honest limits
* Rainbow paper (Atari Space Invaders, 200M frames, no-op starts): DQN 1 692, Rainbow 18 789. Our game is a
  re-implementation with different rules and our budget is ~1–2% of the paper's, so the numbers are not comparable;
  the meaningful comparison is **our agent vs random on the same game and seeds**.
* Reward clipping makes all kills worth +1, so the agent does not prefer 30-point invaders.
* Time-limit truncation is stored as a terminal transition (small bias, as in most DQN implementations).